# Multi-HARM v3.1 — Kaggle run (T4)Reduced **real-data** demo: 400 clean MS-MARCO + 400 injected (4 attack types × 5goals × 20) = 800 samples on Llama-3.1-8B-Instruct at 4-bit. Same pipeline and samecode as the full 2,000-sample run; only the sizes in cell 2 differ.**Before running anything** (Kernel settings panel):- Accelerator → **Nvidia T4 x2**- Internet and speed → **Internet enabled**- Environment variables → add secret **`HF_TOKEN`** (Llama-3.1 is gated) — or use the  ungated model swap in cell 2Details and the resume recipe: `KAGGLE.md`. Cells 3–4 are CPU-only and take ~3 mintotal; run them *before* spending GPU time — they are what caught the bug that madeevery v3.0 AUROC a constant 0.5.

In [ ]:
!pip install -q "transformers>=4.45" accelerate bitsandbytes datasets!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"# deliberately NOT `pip install -r requirements.txt`: that re-resolves torch and can# break Kaggle's preinstalled torch/bitsandbytes pairing. See KAGGLE.md §2.

In [ ]:
import os, shutilos.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"try:    from kaggle_secrets import UserSecretsClient    tok = UserSecretsClient().get_secret("HF_TOKEN")    if tok:        from huggingface_hub import login        login(token=tok)        print("HF_TOKEN secret found — gated models available")    else:        print("no HF_TOKEN secret — using the ungated swap below")except Exception as e:    print(f"(no HF secret: {type(e).__name__}) — using the ungated swap below")# ---- run configuration (persists for every later cell) ----DEMO = Trueos.environ.update({    "MULTI_HARM_DATA_DIR": "/kaggle/working/data",    "MULTI_HARM_OUT_DIR": "/kaggle/working/out",    "MULTI_HARM_QUANT": "nf4",})if DEMO:    os.environ.update({"MULTI_HARM_N_CLEAN": "400",                       "MULTI_HARM_N_INJ_PER_CELL": "20",                       "MULTI_HARM_N_BASE_PAIRS": "900"})else:   # full run    os.environ.update({"MULTI_HARM_N_CLEAN": "1000",                       "MULTI_HARM_N_INJ_PER_CELL": "25",                       "MULTI_HARM_N_BASE_PAIRS": "2200",                       "MULTI_HARM_CALIB_PER_SPECIALIST": "400"})if not os.environ.get("MULTI_HARM_MODEL_NAME"):    # comment out if you logged in with HF_TOKEN and want the paper's model    os.environ["MULTI_HARM_MODEL_NAME"] = "microsoft/Phi-3.5-mini-instruct"    # os.environ["MULTI_HARM_MODEL_NAME"] = "meta-llama/Meta-Llama-3.1-8B-Instruct"print("model:", os.environ["MULTI_HARM_MODEL_NAME"])print("data :", os.environ["MULTI_HARM_DATA_DIR"], "| out:", os.environ["MULTI_HARM_OUT_DIR"])

In [ ]:
# Kaggle keeps the kernel process, so os.environ carries into subprocesses.# This helper is how `!source ./demo_env.sh` would be done on Colab.import subprocess, sys, timedef run(script, *args, timeout=8 * 3600):    t0 = time.time()    cmd = [sys.executable, script, *args]    print("$ " + " ".join(cmd), flush=True)    r = subprocess.run(cmd, env=os.environ, timeout=timeout)    print(f"[exit {r.returncode}] {time.time() - t0:.0f}s", flush=True)    assert r.returncode == 0, f"{script} failed — see the traceback above"    return r

In [ ]:
# CELL 3 — no GPU, no downloads: proves stages 02 + 04-11 against a planted# signal cache (each attack type visible only on its own head). ~21 s.run("run_offline_check.py")

In [ ]:
# CELL 4 — still no GPU: proves stages 01 + 03 against a REAL transformer# forward pass, using a tiny random-init Llama-architecture model built locally# (no download). Covers the §2.0 token-range gate, span clipping, the chunked# cache writer, checkpoint resume and the staleness guard. ~3 min.# Numbers here are meaningless by construction — this is a wiring test.run("run_tiny_model_check.py", "--full")

In [ ]:
# STAGE 01 — env report, model download (~5 GB), shape smoke test, AUROC self-testrun("01_setup_and_validate.py")

In [ ]:
# STAGE 02 — dataset build (MS-MARCO clean pool + 4×5×N injected)run("02_build_dataset.py")

In [ ]:
# STAGE 03a — §2.3 quantization comparison (optional, ~30 min): fp-ref vs 4-bit# correlation at L*. Run it BEFORE the long extraction so a bitsandbytes/CUDA# problem shows up while it is still cheap.run("03_extract_signals.py", "--quant-compare")

In [ ]:
# STAGE 03b — §2.0 token-range validation gate, then signal extraction.# THE LONG CELL: 30-75 min at demo sizes, 2-5 h for the full run.# Resumable within this session: if the kernel restarts, re-run this cell and it# skips already-extracted samples (out/progress/extract.json, flushed before every# checkpoint save). It aborts if >5% of samples cannot be tokenized into valid# spans, and prints how many spans were clipped by MAX_SEQ_LEN.run("03_extract_signals.py")

In [ ]:
# STAGES 04-08 — calibration and the meta layer. Seconds each: these read the# cached signals, so no model is loaded and a T4 is not needed.for s in ("04_calibrate_hstar.py", "05_baseline_attn_tracker.py",          "06_calibrate_general.py", "07_calibrate_specialists.py",          "08_meta_decision.py"):    run(s)

In [ ]:
# STAGE 09 — Tables A-E, §4.3 (+ head×type cross-tab), §4.4, the §4.8 spine# table, the span-width audit, the calibration-size sweep, and latency measured# with the forward pass included.run("09_experiments_analysis.py", "--calib-sweep", "--with-model")

In [ ]:
# STAGES 10-11 — figures + paper-facing report, then the reproducibility ziprun("10_figures_report.py")run("11_reproducibility.py")print()for rel in ("out/experiments/SUMMARY.md", "out/report/RESULTS.md",            "out/validation/width_invariance.json",            "out/repro/repro_manifest.json", "out/repro/multi_harm_repro.zip"):    p = os.path.join(os.environ["MULTI_HARM_OUT_DIR"], rel)    print(f"  {'OK ' if os.path.exists(p) else 'MISSING'} {p}"          + (f"  ({os.path.getsize(p) / 1e6:.2f} MB)" if os.path.exists(p) else ""))print("\nFigures:", sorted(os.listdir(os.path.join(os.environ['MULTI_HARM_OUT_DIR'], 'figures'))))

## Reading the output- **`out/experiments/SUMMARY.md`** — every v3 success criterion with MET/NOT MET,  the §4.8 table, the width-confound measurement and the calibration-size sweep.- **`out/report/RESULTS.md`** — the paper-facing report: §4.8 lead table, Table A,  §4.3 head-level specialization, §4.4 honest attribution (Table D's argmax  accuracy is upward-biased by construction — quote both), §4.9 differentiation,  limitations, and an explicit caveat for any specialist whose FPR budget was not  resolvable at the calibration size.- **`out/repro/multi_harm_repro.zip`** — code + every artifact +   `repro_manifest.json` (git commit, SHA-256 of code and artifacts, and  `data/signals/sigcache_meta.json`, i.e. what the cache was built from).## If something fails| Symptom | Cause / action ||---|---|| `libcublasLt... not found`, nf4 load error | torch was reinstalled under Kaggle's CUDA. Re-run cell 1 *without* touching torch, or `!pip install -q --force-reinstall bitsandbytes` || gated repo / 401 | no `HF_TOKEN` secret → use the Phi-3.5 swap in cell 2 || `01` raises "could not determine attention head count" | the model's config uses another attribute name; add it to `_N_HEAD_ATTRS` in `multi_harm_common/model.py` (this is the check that replaced v3.0's mid-run `AttributeError`) || §2.0 gate reports a bad mapping | payload/prompt layout mismatch — fix `chat.py` before extracting; v3.0 extracted 2,000 samples with no gate at all || CUDA OOM in `03` | `MULTI_HARM_MAX_SEQ_LEN=768` (eager attention is quadratic), or `BATCH`-free smaller `N_INJ_PER_CELL` || `data/signals holds N rows extracted from a DIFFERENT dataset` | sizes/model changed since the cache → re-run `02`, then `03 --fresh` || FPR far from 5% | read the `fpr_resolution limited` note first: with ≤80 clean calibration rows per specialist the 1.25% budget is not expressible; raise `MULTI_HARM_CALIB_PER_SPECIALIST` |